In [11]:
import pandas as pd

dictionary = pd.read_excel("../data/raw/Data Dictionary.xls")

dictionary

,Unnamed: 0,Unnamed: 1,Unnamed: 2
0,Variable Name,Description,Type
1,SeriousDlqin2yrs,Person experienced 90 days past due delinquenc...,Y/N
2,RevolvingUtilizationOfUnsecuredLines,Total balance on credit cards and personal lin...,percentage
3,age,Age of borrower in years,integer
4,NumberOfTime30-59DaysPastDueNotWorse,Number of times borrower has been 30-59 days p...,integer
5,DebtRatio,"Monthly debt payments, alimony,living costs di...",percentage
6,MonthlyIncome,Monthly income,real
7,NumberOfOpenCreditLinesAndLoans,Number of Open loans (installment like car loa...,integer
8,NumberOfTimes90DaysLate,Number of times borrower has been 90 days or m...,integer
9,NumberRealEstateLoansOrLines,Number of mortgage and real estate loans inclu...,integer


In [21]:
# загружаем основной датасет и выводим информоцию для первичного анализа
train = pd.read_csv("../data/raw/cs-training.csv")

display(train.head())

print("Размер:", train.shape)

train.info()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


Размер: (150000, 12)
<class 'pandas.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Unnamed: 0                            150000 non-null  int64  
 1   SeriousDlqin2yrs                      150000 non-null  int64  
 2   RevolvingUtilizationOfUnsecuredLines  150000 non-null  float64
 3   age                                   150000 non-null  int64  
 4   NumberOfTime30-59DaysPastDueNotWorse  150000 non-null  int64  
 5   DebtRatio                             150000 non-null  float64
 6   MonthlyIncome                         120269 non-null  float64
 7   NumberOfOpenCreditLinesAndLoans       150000 non-null  int64  
 8   NumberOfTimes90DaysLate               150000 non-null  int64  
 9   NumberRealEstateLoansOrLines          150000 non-null  int64  
 10  NumberOfTime60-89DaysPastDueNotWorse  150000 non-null  int

In [22]:
# собираем подробную информацию
print("Количество дубликатов:", train.duplicated().sum())

print("\nРаспределение целевой переменной:")
print(train['SeriousDlqin2yrs'].value_counts())

print('\nДоля классов:')
print(train['SeriousDlqin2yrs'].value_counts(normalize=True))

print('\nПропуски:')
missing = train.isna().sum()
missing_pct = train.isna().mean() * 100

missing_table = (
    pd.DataFrame({
        'missing_count':missing, 
        'missing_pct': missing_pct
    }).sort_values('missing_count', ascending=False)
)

display(missing_table)

Количество дубликатов: 0

Распределение целевой переменной:
SeriousDlqin2yrs
0    139974
1     10026
Name: count, dtype: int64

Доля классов:
SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64

Пропуски:


,missing_count,missing_pct
MonthlyIncome,29731,19.820667
NumberOfDependents,3924,2.616000
Unnamed: 0,0,0.000000
SeriousDlqin2yrs,0,0.000000
age,0,0.000000
RevolvingUtilizationOfUnsecuredLines,0,0.000000
DebtRatio,0,0.000000
NumberOfTime30-59DaysPastDueNotWorse,0,0.000000
NumberOfOpenCreditLinesAndLoans,0,0.000000
NumberOfTimes90DaysLate,0,0.000000


In [23]:
# статистический аудит данных 
train.describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,150000.0,75000.500000,43301.414527,1.0,37500.750000,75000.500000,112500.250000,150000.0
SeriousDlqin2yrs,150000.0,0.066840,0.249746,0.0,0.000000,0.000000,0.000000,1.0
RevolvingUtilizationOfUnsecuredLines,150000.0,6.048438,249.755371,0.0,0.029867,0.154181,0.559046,50708.0
age,150000.0,52.295207,14.771866,0.0,41.000000,52.000000,63.000000,109.0
NumberOfTime30-59DaysPastDueNotWorse,150000.0,0.421033,4.192781,0.0,0.000000,0.000000,0.000000,98.0
DebtRatio,150000.0,353.005076,2037.818523,0.0,0.175074,0.366508,0.868254,329664.0
MonthlyIncome,120269.0,6670.221237,14384.674215,0.0,3400.000000,5400.000000,8249.000000,3008750.0
NumberOfOpenCreditLinesAndLoans,150000.0,8.452760,5.145951,0.0,5.000000,8.000000,11.000000,58.0
NumberOfTimes90DaysLate,150000.0,0.265973,4.169304,0.0,0.000000,0.000000,0.000000,98.0
NumberRealEstateLoansOrLines,150000.0,1.018240,1.129771,0.0,0.000000,1.000000,2.000000,54.0


In [24]:
# находим масштабы аномалий в данных
suspicious_columns = [
    "age",
    "RevolvingUtilizationOfUnsecuredLines",
    "DebtRatio",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents"
]

for column in suspicious_columns:
    print(f"\n{column}")
    print("Количество уникальных значений:", train[column].nunique())
    print("Количество подозрительно больших значений:")

    if column == "age":
        print((train[column] == 0).sum())

    elif column in [
        "NumberOfTime30-59DaysPastDueNotWorse",
        "NumberOfTimes90DaysLate",
        "NumberOfTime60-89DaysPastDueNotWorse"
    ]:
        print(train[column].isin([96, 98]).sum())

    else:
        print((train[column] > train[column].quantile(0.99)).sum())


age
Количество уникальных значений: 86
Количество подозрительно больших значений:
1

RevolvingUtilizationOfUnsecuredLines
Количество уникальных значений: 125728
Количество подозрительно больших значений:
1500

DebtRatio
Количество уникальных значений: 114194
Количество подозрительно больших значений:
1500

NumberOfTime30-59DaysPastDueNotWorse
Количество уникальных значений: 16
Количество подозрительно больших значений:
269

NumberOfTimes90DaysLate
Количество уникальных значений: 19
Количество подозрительно больших значений:
269

NumberOfTime60-89DaysPastDueNotWorse
Количество уникальных значений: 13
Количество подозрительно больших значений:
269

NumberRealEstateLoansOrLines
Количество уникальных значений: 28
Количество подозрительно больших значений:
1482

NumberOfDependents
Количество уникальных значений: 13
Количество подозрительно больших значений:
991


In [25]:
display(
    train[train["age"] == 0]
)

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
65695,65696,0,1.0,0,1,0.436927,6000.0,6,0,2,0,2.0


In [26]:
# Распределение target среди записей со значениями 96/98
late_columns = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse"
]

for column in late_columns:
    print(f"\n{column}")
    display(
        train[column]
        .value_counts()
        .sort_index()
    )


NumberOfTime30-59DaysPastDueNotWorse


NumberOfTime30-59DaysPastDueNotWorse
0     126018
1      16033
2       4598
3       1754
4        747
5        342
6        140
7         54
8         25
9         12
10         4
11         1
12         2
13         1
96         5
98       264
Name: count, dtype: int64


NumberOfTimes90DaysLate


NumberOfTimes90DaysLate
0     141662
1       5243
2       1555
3        667
4        291
5        131
6         80
7         38
8         21
9         19
10         8
11         5
12         2
13         4
14         2
15         2
17         1
96         5
98       264
Name: count, dtype: int64


NumberOfTime60-89DaysPastDueNotWorse


NumberOfTime60-89DaysPastDueNotWorse
0     142396
1       5731
2       1118
3        318
4        105
5         34
6         16
7          9
8          2
9          1
11         1
96         5
98       264
Name: count, dtype: int64

In [27]:
# Насколько часто 96/98 встречаются одновременно
mask_96_98 = train[late_columns].isin([96, 98]).any(axis=1)

print("Записей с 96/98 хотя бы в одном признаке:", mask_96_98.sum())

display(
    train.loc[mask_96_98, late_columns + ["SeriousDlqin2yrs"]]
    .head(20)
)

Записей с 96/98 хотя бы в одном признаке: 269


,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTimes90DaysLate,NumberOfTime60-89DaysPastDueNotWorse,SeriousDlqin2yrs
1733,98,98,98,1
2286,98,98,98,0
3884,98,98,98,0
4417,98,98,98,0
4705,98,98,98,0
5073,98,98,98,0
6280,98,98,98,1
7032,98,98,98,1
7117,98,98,98,1
7687,98,98,98,1


In [28]:
mask_96_98 = train[late_columns].isin([96, 98]).any(axis=1)

display(
    train.loc[
        mask_96_98,
        [
            "SeriousDlqin2yrs",
            "age",
            "RevolvingUtilizationOfUnsecuredLines",
            "DebtRatio",
            "MonthlyIncome",
            "NumberOfOpenCreditLinesAndLoans",
            "NumberRealEstateLoansOrLines",
            "NumberOfDependents"
        ]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
SeriousDlqin2yrs,269.0,0.546468,4.987639e-01,0.0,0.0,1.0,1.0,1.0
age,269.0,34.245353,1.306182e+01,21.0,24.0,29.0,43.0,79.0
RevolvingUtilizationOfUnsecuredLines,269.0,1.000000,1.112292e-16,1.0,1.0,1.0,1.0,1.0
DebtRatio,269.0,5.739667,2.601902e+01,0.0,0.0,0.0,0.0,255.0
MonthlyIncome,148.0,2557.135135,2.745778e+03,0.0,1333.0,2168.5,3174.5,28733.0
NumberOfOpenCreditLinesAndLoans,269.0,0.007435,8.606510e-02,0.0,0.0,0.0,0.0,1.0
NumberRealEstateLoansOrLines,269.0,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0
NumberOfDependents,238.0,0.369748,8.304901e-01,0.0,0.0,0.0,0.0,5.0


In [29]:
print(
    train.loc[mask_96_98, "SeriousDlqin2yrs"]
    .value_counts(normalize=True)
)

SeriousDlqin2yrs
1    0.546468
0    0.453532
Name: proportion, dtype: float64
